# **Lab 5. Working with Vector Data using GeoPandas**
**GY674 / GY675**

**Today:** Learn how to work with vector data in Python, using Irish Electoral Division boundaries, deprivation data and railway infrastructure to explore geographical patterns of inequality.

### What we'll learn

1. **Read** → Import vector data: points, lines and polygons.
2. **Explore** → Examine spatial features, attributes and coordinate reference systems (CRS).
3. **Merge** → Combine Pobal deprivation data with Electoral Division boundaries using `.merge`
4. **Analyse** → Aggregate Electoral Divisions into counties using `.dissolve()`.
5. **Visualise** → Create maps combining deprivation, railway lines and station points.

### Today's workflow

- **Part 1:** Explore vector data and merge deprivation data to Electoral Divisions.
- **Part 2:** Aggregate Electoral Divisions into counties and map deprivation.
- **Further:** Combine polygons, lines and points on one map.

### How to use this notebook

- 👀 **Demo** – Follow along.
- ✏️ **Your turn** – Practise the code.
- ⭐ **Challenge** – Apply your skills independently.



---
# **Part 1: How does deprivation vary across Ireland at the Electoral Division (ED) level (making our first map)**


# **Step 1. READ**

### Today's datasets

We'll work with two datasets covering Ireland's **Electoral Divisions (EDs)**:

1. **[hp-deprivation-index-scores-2022_lab3.csv](https://github.com/shaunharrigan/maynooth-geo-python/blob/main/data/hp-deprivation-index-scores-2022_lab3.csv)** — a CSV containing deprivation scores for each ED, which we've used before (**Tabular data -> Pandas -> DataFrame**).
2. **[CSO_Electoral_Divisions_National_Statistical_Boundaries_2022.gpkg](https://github.com/shaunharrigan/maynooth-geo-python/releases/tag/datasets-2026)** — a spatial dataset containing the geographical boundaries of EDs (**Vector spatial data (polygons) -> GeoPandas -> GeoDataFrame**).

We'll use **pandas** to read the deprivation data and **GeoPandas** to read the spatial boundaries. Later, we'll join them together to create maps.

### **Really Important to get the set-up right:**

* Download and saveboth datasets above (.csv & .gpkg) in your Google Drive folder
* Import all packages needed
* Connect Colab to Google Drive
* Read the CSV using `pd.read_csv()` and the spatial data using `gpd.read_file()`
  
**Remember:** Your file paths and filenames must match where you saved the data.

In [ ]:
# Import packages
import pandas as pd          # Work with tabular data
import geopandas as gpd       # Work with spatial (vector) data
import matplotlib.pyplot as plt  # Create plots and maps

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the data directory
data_dir = "/content/drive/MyDrive/geo_python_labs_2026/data/"

# Define the output directory, e.g. where to save Figures and new datasets
output_dir = "/content/drive/MyDrive/geo_python_labs_2026/outputs/"

# Read the Pobal CSV (update the filename to match your file)
pobal_df = pd.read_csv(data_dir + "hp-deprivation-index-scores-2022_lab3.csv")

# Read Electoral Division boundaries
ed_gdf = gpd.read_file(data_dir + "CSO_Electoral_Divisions_National_Statistical_Boundaries_2022.gpkg")

# **Step 2. EXPLORE**

In [ ]:
# Have a look again at the pobal dataset
pobal_df

In [ ]:
# Look at the type...
print(type(pobal_df))

In [ ]:
# Have a look at the data types etc (it's a Pandas DataFrame)
pobal_df.info()

In [ ]:
# Now let's check out the Electoral Division (ED) vector dataset (it's a GeoDataFrame)
ed_gdf

In [ ]:
# Check out .info()
ed_gdf.info()

In [ ]:
# Check the dataset type...
print(type(ed_gdf))

In [ ]:
# Chek number of rows and columns
print("Shape:", ed_gdf.shape)

In [ ]:
# Check column names
print("Columns:", ed_gdf.columns)

In [ ]:
# Check the spatial coordinate reference system (CRS)
# Coordinate reference system
print("Coordinate Reference System (CRS) is:", ed_gdf.crs)

In [ ]:
# Quick map
ed_gdf.plot()

## Check your understanding

* Have you seen that the Pobal .csv data is read as a Pandas DataFrame?
* Have you seen that the ED vector data (.gpkg) is read as a GeoPandas GeoDataFrame?

# **Step 3. PREPARE**

Our next task is to **merge two datasets** so we can analyse and map deprivation across Ireland's Electoral Divisions (EDs).

We'll first use a small example to introduce pandas `.merge()`, which allows us to combine datasets using a common identifier.

We'll then apply this to our real datasets:

- **ED boundaries (GeoDataFrame):** Contains the geographical boundaries and attributes of each Electoral Division.
- **Pobal deprivation data (DataFrame):** Contains deprivation information for each Electoral Division.

**Our goal:** Add the Pobal deprivation data to the attribute table of our ED boundaries, so we can map and explore geographical patterns of deprivation.


### 👀 Demo | Merging datasets with `.merge()`

We can use pandas `.merge()` to combine two datasets using a **common column**.

There are four main types of merges, depending on which records we want to keep:

- **Left:** Keep all rows from the first dataset.
- **Right:** Keep all rows from the second dataset.
- **Inner:** Keep only rows with matching values in both datasets.
- **Outer:** Keep all rows from both datasets.

<img src="https://raw.githubusercontent.com/shaunharrigan/maynooth-geo-python/main/images/pandas_merge_types.png" width="750">

Let's try a **left merge** and an **inner merge** using two small datasets.

We have two simple pandas DataFrames:

- **Dataset 1:** ED information (ID, name and county).
- **Dataset 2:** Deprivation scores for each ED.

Both datasets contain a common column, `ED_ID_STR`, which we'll use to merge them.

In [ ]:
# Dataset 1: ED information
eds = pd.DataFrame({
    "ED_ID_STR": ["001", "002", "003"],
    "ED_Name": ["North", "South", "West"],
    "County": ["Donegal", "Mayo", "Galway"]
})

# Dataset 2: Deprivation scores
deprivation = pd.DataFrame({
    "ED_ID_STR": ["001", "002", "004"],
    "Deprivation": [-5.2, 1.3, 0.8]
})

# Display both datasets
display(eds, deprivation)

In [ ]:
# Inner join: keep only matching EDs
merged_inner = eds.merge(deprivation, on="ED_ID_STR", how="inner")

merged_inner

In [ ]:
# Inner join: keep only matching EDs

# Try a left join, create a new left join on eds with deprivation on="ED_ID_STR" called 'merged_left'
merged_left = eds.merge(deprivation, on="ED_ID_STR", how="left")

merged_left


In [ ]:
# Now check for any missing values (i.e. NA's or NaN's)
merged_left.isna().sum()


### 👀 Demo | Merging our ED boundaries with Pobal deprivation data

Now let's apply `.merge()` to our real datasets.

We'll use a **left merge** to add the Pobal deprivation data (`pobal_df`) to our ED boundaries (`ed_gdf`), matching records using `ED_ID_STR`.

Our goal is to **retain all ED boundaries** and add their corresponding deprivation information, so we can map deprivation across Ireland.


In [ ]:
# Left merge: keep all ED boundaries within 'ed_gdf'
merged_ed_gdf = ed_gdf.merge(pobal_df, on="ED_ID_STR", how="left")

merged_ed_gdf

In [ ]:
# Check that there's not mssing values to ensure the 'merge' worked.
merged_ed_gdf.isna().sum()

### 🔧 Fixing our merge | Why are values missing?

Our merge retained all ED boundaries, but around 1,400 EDs have missing deprivation scores. Why?

The problem is with our **ED identifiers (`ED_ID_STR`)**. These identifiers should contain **six digits**, but some have lost their leading zeros in the original Pobal CSV.

For example, `048039` appears as `48039`, meaning it no longer matches the identifier in our ED boundaries.

We can fix this using **zero-padding**: adding zeros to the beginning of a string until it reaches the required length.

Pandas provides a simple method for this: [`.str.zfill()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.zfill.html).

Let's reload the Pobal data, pad the identifiers to six characters, and try our merge again.

In [ ]:
# Reload the original Pobal data
pobal_df = pd.read_csv(data_dir + "hp-deprivation-index-scores-2022_lab3.csv")

# Create a copy and pad ED identifiers with leading zeros
pobal_df_fixed = pobal_df.copy()
pobal_df_fixed["ED_ID_STR"] = pobal_df_fixed["ED_ID_STR"].str.zfill(6)

In [ ]:
# Now check ED ID's have 6 characters with zeros at front for those less than 6
pobal_df_fixed["ED_ID_STR"][0:40]

In [ ]:
# Repeat the left merge using the corrected Pobal data
merged_ed_gdf_fixed = ed_gdf.merge(pobal_df_fixed, on="ED_ID_STR", how="left")

In [ ]:
# Check that there's now no missing rows...
merged_ed_gdf_fixed.isna().sum()

### Selecting columns for our analysis

Our merged GeoDataFrame contains many columns that we don't need. Let's **select only the relevant columns** to create a simpler GeoDataFrame for our analysis and mapping.

We'll keep the following four columns:

| Column | Purpose |
|---|---|
| `ED_ENGLISH_x` | Identify individual Electoral Divisions |
| `COUNTY` | Identify Counties |
| `Index22_ED_std_rel_wt` | Analyse deprivation scores |
| `geometry` | Store the geographical shapes needed for mapping |

We'll save these as a new GeoDataFrame, keeping our original merged dataset unchanged.


In [ ]:
# Select only the columns needed for our analysis
ed_deprivation_gdf = merged_ed_gdf_fixed[
    ["ED_ENGLISH_x", "COUNTY", "Index22_ED_std_rel_wt", "geometry"]
].copy()

# Display our prepared GeoDataFrame
ed_deprivation_gdf.head()

In [ ]:
# Rename columns for clarity
ed_deprivation_gdf = ed_deprivation_gdf.rename(columns={
    "ED_ENGLISH_x": "ED",
    "Index22_ED_std_rel_wt": "Deprivation_score"
})

ed_deprivation_gdf.head()

**CHECK POINT BEFORE MOVING TO STEP 4 and 5**

Let's reset the 'Runtime' and check we have everthing correct:

*  **RESTART THE RUNTIME** by clicking on **Runtime** on top left menu **->** **Restart session**
* **Run all cells in order again cleanly before** this cell. To do this, make sure you select this cell -> click **Runtime** from menu **-> Run before**

---



In [ ]:
# Check our merged dataset is ready for analysis
print("Number of ED polygons:", len(ed_deprivation_gdf))
print("Missing deprivation scores:",
      ed_deprivation_gdf["Deprivation_score"].isna().sum())

# **Step 4. ANALYSE**

Now that we've successfully merged our datasets, we can start exploring **deprivation across Ireland's Electoral Divisions (EDs)**.

Before creating a map, let's examine the deprivation scores using some familiar pandas methods.

We'll:

1. **Summarise** the deprivation scores using `.describe()`.
2. **Explore** their distribution using a histogram.
3. **Identify** the EDs with the highest and lowest deprivation scores.

This will help us understand the data before we visualise its geographical patterns.

**Key question:** How much does deprivation vary between Electoral Divisions in Ireland?

In [ ]:
# Summary statistics for ED deprivation scores
ed_deprivation_gdf["Deprivation_score"].describe()

In [ ]:
# Plot a histogram of ED deprivation scores
ed_deprivation_gdf["Deprivation_score"].hist(bins=30)

In [ ]:
# Most deprived and most affluent EDs
ed_deprivation_gdf.sort_values("Deprivation_score")

# **Step 5. VISUALISE**

Now let's create our first **choropleth map** — a map where geographical areas are coloured according to a data value.

We'll map deprivation scores across Ireland's Electoral Divisions (EDs) using GeoPandas `.plot()`.

But making a map is only the beginning. We'll progressively improve its design by considering:

1. **Colour:** Which colours best communicate deprivation and affluence?
2. **Scale:** How should we represent negative and positive scores?
3. **Presentation:** How can we make our map clear, informative and visually appealing?

**Key question:** How can thoughtful map design help us communicate geographical patterns more effectively?


**Map 1 — Our first choropleth map**

In [ ]:
# Our first deprivation map
ed_deprivation_gdf.plot(
    column="Deprivation_score",
    legend=True,
    figsize=(8, 8)
)

**Choosing a colour palette**

Good maps use colours that help communicate the data. Let's explore **sequential** and **diverging** colour palettes, and how `cmap`, `vmin` and `vmax` control their appearance.

<img src="https://raw.githubusercontent.com/shaunharrigan/maynooth-geo-python/main/images/quick_colormaps_cheat_sheet.png" width="900">

**For our deprivation map:** Why is a diverging colour palette centred on zero more appropriate?

In [ ]:
# Improve the colour palette
ed_deprivation_gdf.plot(
    column="Deprivation_score",
    cmap="RdBu",
    legend=True,
    figsize=(8, 8)
)

**Need to set sensible vmin and vmax...**

In [ ]:
# Map deprivation scores with a colour scale from -15 to +15
ed_deprivation_gdf.plot(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-15,
    vmax=15,
    legend=True,
    figsize=(8, 8)
)

### **Now make a few aesthtic tweaks and save our map to our Google Drive as a .png**

In [ ]:
# Create figure and axes
fig, ax = plt.subplots(figsize=(8, 9))

# Map ED deprivation scores
ed_deprivation_gdf.plot(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-15,
    vmax=15,
    legend=True,
    legend_kwds={"label": "Deprivation score", "shrink": 0.6},
    ax=ax
)

# Improve map presentation
ax.set_title("Deprivation across Ireland's Electoral Divisions (2022)")
ax.set_axis_off()

# Save the map_
fig.savefig(output_dir + "ireland_ed_deprivation.png", dpi=300, bbox_inches="tight")

plt.show()

---

# **Part 2: How does deprivation vary across Ireland at the County level?**

We've mapped deprivation across Electoral Divisions (EDs). But what if we want to explore deprivation at a **larger geographical scale**?

GeoPandas `.dissolve()` combines smaller polygons into larger areas, while allowing us to summarise their data (e.g. calculate a mean).

<img src="https://raw.githubusercontent.com/shaunharrigan/maynooth-geo-python/main/images/geopandas_dissolve.png" width="800">

We'll **dissolve EDs into Counties**, calculate their mean deprivation scores, and create a new map.

**Key question:** How do deprivation patterns change when we move from EDs to Counties?


### Dissolving EDs into Counties

We'll use GeoPandas `.dissolve()` to combine ED polygons into larger County areas.

- **`by="COUNTY"`** groups EDs by County and merges their geometries.
- **`aggfunc={"Deprivation_score": "mean"}`** specifies which variable to aggregate and how — calculating the mean deprivation score for each County.

This is similar to pandas `.groupby()`, but also combines the geographical shapes.

**Further reading:** [GeoPandas documentation — Dissolving geometries](https://geopandas.org/en/stable/docs/user_guide/aggregation_with_dissolve.html)


In [ ]:
# Have a look at the ed deprivation by ED GeoDataFrame data again
ed_deprivation_gdf

In [ ]:
### THIS CAN TAKE A MINUTE OR SO TO RUN (a lot of calculations) ###

# Dissolve EDs into Counties and calculate mean deprivation
mean_depriv_county = _gdf = ed_deprivation_gdf.dissolve(
    by="COUNTY",
    aggfunc={"Deprivation_score": "mean"}
)

mean_depriv_county

In [ ]:
# Sort counties from low to high, like we did in Lab 3
mean_depriv_county["Deprivation_score"].sort_values()

## **Now, let map the mean deprivation at county level**

**Explore:** [Matplotlib colormap gallery](https://matplotlib.org/stable/gallery/color/colormap_reference.html) — try changing `cmap="RdBu"` to `"coolwarm"`, `"BrBG"` or `"viridis"` and see how the map changes.

In [ ]:
# Create figure and axes
fig, ax = plt.subplots(figsize=(8, 9))

# Map mean County deprivation scores
mean_depriv_county.plot(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-2,
    vmax=2,
    legend=True,
    legend_kwds={"label": "Mean deprivation score", "shrink": 0.6},
    linewidth=0.5,
    edgecolor="white",
    ax=ax
)

# Improve map presentation
ax.set_title("Mean deprivation score by County (2022)")
ax.set_axis_off()

# Save the map
fig.savefig(output_dir + "ireland_county_mean_deprivation.png", dpi=300, bbox_inches="tight")

plt.show()

### **Interactive maps with `.explore()`**

GeoPandas `.explore()` creates **interactive maps** that allow us to zoom, pan and explore geographical data.

Unlike `.plot()`, which creates static maps, `.explore()` uses Folium to create interactive web maps.

Try changing the colourmap (`cmap`) to explore different options.

📖 **Further reading:** [GeoPandas documentation — Interactive mapping](https://geopandas.org/en/stable/docs/user_guide/interactive_mapping.html)


In [ ]:
# Interactive County deprivation map (can take 30secs to a minute to run)
mean_depriv_county.explore(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-2,
    vmax=2,
    tiles=None
)

# **Further & Recomended homework: Combine polygons, lines and points on one map**


## **Average county deprivation and national rail infrastructure**

We will combine **polygons, lines and points** to explore geographical patterns in deprivation and transport infrastructure across Ireland.

- **Polygons:** Mean county deprivation scores calculated earlier.
- **Lines:** [Rail Network – National 250k Map of Ireland](https://data.gov.ie/en_GB/dataset/rail-network-national-250k-map-of-ireland), published by **Tailte Éireann**.
- **Points:** Railway stations, also from Tailte Éireann.
- Combine all three datasets in **GeoPandas** to create a single map.

The railway datasets are loaded directly from an online [ArcGIS Feature Service](https://services-eu1.arcgis.com/FH5XCsx8rYXqnjF5/ArcGIS/rest/services), using URLs provided in the code.

### 💡 A note on coordinate reference systems (CRS)

Different spatial datasets may use different **coordinate reference systems (CRS)**, which define how locations are represented on a map.

Before combining datasets, we need to ensure they use the same CRS.

```python
# Check coordinate systems
print(mean_depriv_county.crs)
print(rail_gdf.crs)
print(stations_gdf.crs)

# Convert to match the county boundaries
rail_gdf = rail_gdf.to_crs(mean_depriv_county.crs)
stations_gdf = stations_gdf.to_crs(mean_depriv_county.crs)
```

**Remember:** `.to_crs()` transforms coordinates so that different spatial layers align correctly.

[Further reading: GeoPandas – Managing projections](https://geopandas.org/en/stable/docs/user_guide/projections.html)


In [ ]:
# Read railway network directly from ArcGIS
url = (
    "https://services-eu1.arcgis.com/FH5XCsx8rYXqnjF5/"
    "ArcGIS/rest/services/"
    "Rail_Network___OSi_National_250k_Map_Of_Ireland/"
    "FeatureServer/0/query"
    "?where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

rail_gdf = gpd.read_file(url)

# Explore the data
rail_gdf.plot()

In [ ]:
# Add to County deprivation map
# Add railway network to county deprivation map
fig, ax = plt.subplots(figsize=(8, 9))

# County deprivation
mean_depriv_county.plot(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-2,
    vmax=2,
    legend=True,
    legend_kwds={"label": "Mean deprivation score", "shrink": 0.6},
    linewidth=0.5,
    edgecolor="white",
    ax=ax
)

# Railway network
rail_gdf.to_crs(mean_depriv_county.crs).plot(
    ax=ax,
    color="darkgreen",
    linewidth=3
)

ax.set_title("Deprivation and Railway Network in Ireland")
ax.set_axis_off()

plt.show()

## **Combining polygons, lines and points**

- **Polygons:** County boundaries coloured by mean deprivation score.
- **Lines:** National railway network showing transport infrastructure.
- **Points:** [Railway stations](https://services-eu1.arcgis.com/FH5XCsx8rYXqnjF5/ArcGIS/rest/services/Railway_Stations___OSi_National_250k_Map_Of_Ireland/FeatureServer) from Tailte Éireann's ArcGIS API.
- **Combine all three layers** in GeoPandas, ensuring they share the same CRS.
- **Explore:** How does railway infrastructure vary across counties with different levels of deprivation?

In [ ]:
# Read railway stations (point data) directly from ArcGIS API
url = (
    "https://services-eu1.arcgis.com/FH5XCsx8rYXqnjF5/"
    "ArcGIS/rest/services/"
    "Railway_Stations___OSi_National_250k_Map_Of_Ireland/"
    "FeatureServer/0/query"
    "?where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

stations_gdf = gpd.read_file(url)

# Explore the data
stations_gdf.plot()

In [ ]:
# Combine polygons, lines and points on one map
fig, ax = plt.subplots(figsize=(8, 9))

# 1. Polygons: Mean county deprivation
mean_depriv_county.plot(
    column="Deprivation_score",
    cmap="RdBu",
    vmin=-2,
    vmax=2,
    legend=True,
    legend_kwds={"label": "Mean deprivation score", "shrink": 0.6},
    linewidth=0.5,
    edgecolor="white",
    zorder=1,
    ax=ax
)

# 2. Lines: Railway network
rail_gdf.to_crs(mean_depriv_county.crs).plot(
    ax=ax,
    color="darkgreen",
    linewidth=3,
    zorder=2
)

# 3. Points: Railway stations (on top)
stations_gdf.to_crs(mean_depriv_county.crs).plot(
    ax=ax,
    color="limegreen",
    markersize=40,
    edgecolor="white",
    linewidth=0.7,
    zorder=3
)

ax.set_title("County Deprivation and Railway Infrastructure in Ireland")
ax.set_axis_off()

plt.show()

# **End**